In [ ]:
# Training the ResNeXt 3D CNN with stronger regularisation 
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.utils import to_categorical

# Load the training dataset
X_train = np.load('/path/to/training/data/X_train.npy')  # Replace with your actual path
Y_train = np.load('/path/to/training/data/Y_train.npy')  # Replace with your actual path

# Add depth dimension if needed (e.g., from shape (529, 26, 21, 3) to (529, 26, 21, 3, 1))
X_train = X_train[..., np.newaxis]  # Add an additional channel dimension if needed

# Convert labels to one-hot encoding if needed
Y_train = to_categorical(Y_train)  # Adjust if labels are already one-hot encoded

# Define the ResNeXt 3D CNN architecture with stronger regularization
def build_resnext_3d(input_shape, num_classes):
    weight_decay = 0.01  # Stronger L2 regularization

    model = models.Sequential()
    model.add(layers.InputLayer(input_shape=input_shape))

    # First Convolutional Block
    model.add(layers.Conv3D(filters=64, kernel_size=(3, 3, 3), padding='same', 
                            strides=(1, 1, 1), kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Dropout(0.3))  # Increased dropout

    model.add(layers.Conv3D(filters=64, kernel_size=(3, 3, 3), padding='same', 
                            groups=8, kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Dropout(0.3))

    # Max Pooling
    model.add(layers.MaxPooling3D(pool_size=(2, 2, 2), padding='same'))

    # Second Convolutional Block
    model.add(layers.Conv3D(filters=128, kernel_size=(3, 3, 3), padding='same', 
                            strides=(1, 1, 1), kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Dropout(0.4))  # Stronger dropout

    model.add(layers.Conv3D(filters=128, kernel_size=(3, 3, 3), padding='same', 
                            groups=8, kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Dropout(0.4))

    # Max Pooling
    model.add(layers.MaxPooling3D(pool_size=(2, 2, 2), padding='same'))

    # Flatten and Fully Connected Layers
    model.add(layers.Flatten())

    model.add(layers.Dense(512, kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(layers.ReLU())
    model.add(layers.Dropout(0.5))  # Stronger dropout

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax', kernel_regularizer=regularizers.l2(weight_decay)))

    return model

# Set input shape and number of classes
input_shape = X_train.shape[1:]  # (26, 21, 3, 1) after adding depth
num_classes = Y_train.shape[1]   # Number of unique classes

# Build and compile the model
model = build_resnext_3d(input_shape=input_shape, num_classes=num_classes)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, Y_train, batch_size=32, epochs=20, validation_split=0.2)


In [ ]:
import tensorflow as tf
import os

# Function to print model statistics
def print_model_stats(model):
    print("\n📊 Model Statistics:")
    model.summary()  # Prints layer-wise model summary
    total_params = model.count_params()
    trainable_params = sum(p.count for p in model.trainable_variables)
    non_trainable_params = total_params - trainable_params
    print(f"\n🔹 Total Parameters: {total_params:,}")
    print(f"🔹 Trainable Parameters: {trainable_params:,}")
    print(f"🔹 Non-Trainable Parameters: {non_trainable_params:,}")

# Function to save model and print directory path
def save_model_and_print_path(model, save_dir="saved_model"):
    os.makedirs(save_dir, exist_ok=True)  # Create directory if not exists
    model_path = os.path.join(save_dir, "resnext_3d_model")
    model.save(model_path)
    print(f"\n✅ Model saved at: {os.path.abspath(model_path)}")

# Assuming 'model' is already trained
print_model_stats(model)
save_model_and_print_path(model)


In [ ]:
import tensorflow as tf
import os

# Function to print model statistics
def print_model_stats(model):
    print("\n📊 Model Statistics:")
    model.summary()  # Prints layer-wise model summary
    total_params = model.count_params()
    trainable_params = sum(tf.size(p).numpy() for p in model.trainable_variables)
    non_trainable_params = total_params - trainable_params
    print(f"\n🔹 Total Parameters: {total_params:,}")
    print(f"🔹 Trainable Parameters: {trainable_params:,}")
    print(f"🔹 Non-Trainable Parameters: {non_trainable_params:,}")

# Function to save model and print directory path
def save_model_and_print_path(model, save_dir="saved_model"):
    os.makedirs(save_dir, exist_ok=True)  # Create directory if not exists
    model_path = os.path.join(save_dir, "resnext_3d_model")
    model.save(model_path)
    print(f"\n✅ Model saved at: {os.path.abspath(model_path)}")

# Assuming 'model' is already trained
print_model_stats(model)
save_model_and_print_path(model)


In [ ]:
import tensorflow as tf
import os

# Function to save the model and print the directory path
def save_model_and_print_path(model, save_dir="saved_model"):
    os.makedirs(save_dir, exist_ok=True)  # Ensure directory exists
    
    model_path = os.path.join(save_dir, "resnext_3d_model.keras")  # Append .keras extension
    model.save(model_path)  # Save the model in the recommended format
    print(f"\n✅ Model saved at: {os.path.abspath(model_path)}")

# Assuming 'model' is already trained
save_model_and_print_path(model)


In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical

# Load the training data
X_train = np.load('/path/to/training/data/X_train.npy')
Y_train = np.load('/path/to/training/data/Y_train.npy')

# Reshape input if needed
X_train = X_train[..., np.newaxis]  # Shape: (samples, 26, 21, 3, 1)

# One-hot encode labels if not already done
Y_train = to_categorical(Y_train)

# Load the previously saved model
model = load_model('/path/to/trained/resnext_3d_model.keras')

# Continue training for more epochs
model.fit(X_train, Y_train, batch_size=32, epochs=10, validation_split=0.2)


In [ ]:
# Save the updated model (overwrite the previous one)
model.save('/Users/2018289369/Documents/Model training/saved_model/ResNeXt3D_model_updated.keras')


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error

# Load the validation dataset
X_val = np.load('/path/to/validation/data/X_val.npy')  # Replace with actual path
Y_val = np.load('/path/to/validation/data/Y_val.npy')  # Replace with actual path

# Add depth dimension if needed (ensure consistency with training data)
X_val = X_val[..., np.newaxis]  # Shape should be (157, 26, 21, 3, 1) if needed

# Load the trained model (make sure to replace with the actual model file if saved)
model = tf.keras.models.load_model('/path/to/trained/resnext_3d_model.keras')  # Replace with actual path if needed

# Make predictions on the validation set
Y_pred = model.predict(X_val)

# Compute Squared Error for each sample
squared_errors = np.square(Y_pred - Y_val)

# Compute Mean Squared Error (MSE)
mse = np.mean(squared_errors)

print(f"Mean Squared Error on Validation Set: {mse}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error

# Load the validation dataset
X_val = np.load('/path/to/validation/data/X_val.npy')  # Replace with actual path
Y_val = np.load('/Users/2018289369/Documents/Model training/Data/Split_Data/Y_val.npy')  # Replace with actual path

# Add depth dimension if needed
X_val = X_val[..., np.newaxis]  # Ensure shape consistency with training data

# Load the trained model
model = tf.keras.models.load_model('/path/to/trained/resnext_3d_model.keras')  # Replace with actual path

# Make predictions
Y_pred = model.predict(X_val)

# If predictions are one-hot encoded, convert them to scalar labels
if Y_pred.shape[1] > 1:  # Check if multi-class (one-hot)
    Y_pred = np.argmax(Y_pred, axis=1)  # Convert to class labels

# Compute Squared Error for each sample
squared_errors = np.square(Y_pred - Y_val)

# Compute Mean Squared Error (MSE)
mse = np.mean(squared_errors)

print(f"Mean Squared Error on Validation Set: {mse}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load the trained model
model = tf.keras.models.load_model("/path/to/trained/resnext_3d_model.keras")  # Replace with actual model path

# Load the training dataset
X_train = np.load("/path/to/training/data/X_train.npy")  # Replace with actual file path
Y_train = np.load("/path/to/training/data/Y_train.npy")  # Replace with actual file path

# Ensure X_train has the correct shape (add depth dimension if necessary)
X_train = X_train[..., np.newaxis]  # If needed, expand dimensions

# Predict Y values for the training dataset
Y_pred = model.predict(X_train)

# If predictions are one-hot encoded, convert them to scalar labels
if Y_pred.shape[1] > 1:  # Check if multi-class (one-hot)
    Y_pred = np.argmax(Y_pred, axis=1)  # Convert to class labels

# Calculate Squared Error Cost Function: sum of squared differences
squared_error = np.sum((Y_pred - Y_train) ** 2)

print(f"Squared Error Cost on Training Set: {squared_error:.4f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error

# Load the testing dataset
X_test = np.load('/path/to/testing/data/X_test.npy')  # Replace with actual path
Y_test = np.load('/path/to/testing/data/Y_test.npy')  # Replace with actual path

# Add depth dimension if needed
X_test = X_test[..., np.newaxis]  # Ensure shape consistency with training data

# Load the trained model
model = tf.keras.models.load_model('/path/to/trained/resnext_3d_model.keras')  # Replace with actual path

# Make predictions
Y_pred = model.predict(X_test)

# If predictions are one-hot encoded, convert them to class labels
if Y_pred.shape[1] > 1:  # Check if multi-class (one-hot)
    Y_pred = np.argmax(Y_pred, axis=1)  # Convert to scalar labels

# Compute RMSE
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))

print(f"Root Mean Squared Error (RMSE) on Testing Set: {rmse}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import r2_score

# Load the testing dataset
X_test = np.load('/path/to/testing/data/X_test.npy')  # Replace with actual path
Y_test = np.load('/path/to/testing/data/Y_test.npy')  # Replace with actual path

# Add depth dimension if needed
X_test = X_test[..., np.newaxis]  # Ensure shape consistency with training data

# Load the trained model
model = tf.keras.models.load_model('/path/to/trained/resnext_3d_model.keras')  # Replace with actual path

# Make predictions
Y_pred = model.predict(X_test)

# If predictions are one-hot encoded, convert them to class labels
if Y_pred.shape[1] > 1:  # Check if multi-class (one-hot)
    Y_pred = np.argmax(Y_pred, axis=1)  # Convert to scalar labels

# Compute R² Score
r2 = r2_score(Y_test, Y_pred)

print(f"Coefficient of Determination (R²) on Testing Set: {r2:.4f}")
